<a href="https://colab.research.google.com/github/noorarora/ARTI6000-RLHF-Assignment/blob/main/assignment1_rlhf/notebooks/04_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q transformers datasets accelerate sentencepiece torch pandas

In [16]:
import random
import numpy as np
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [17]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Seed set successfully.")

Seed set successfully.


In [18]:
baseline_model_path = "./sft_baseline_distilgpt2"
dpo_model_path = "./dpo_aligned_distilgpt2"
reward_model_path = "./reward_model_distilbert"

print("Baseline model path:", baseline_model_path)
print("DPO model path:", dpo_model_path)
print("Reward model path:", reward_model_path)

Baseline model path: ./sft_baseline_distilgpt2
DPO model path: ./dpo_aligned_distilgpt2
Reward model path: ./reward_model_distilbert


In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [20]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
baseline_model.eval()

print("Baseline model loaded successfully.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline model loaded successfully.


In [22]:
# baseline model
baseline_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

# aligned model (we simulate using the same base model if saved model not present)
dpo_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

# reward model
reward_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
reward_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased").to(device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
test_prompts = [
    "Human: Explain reinforcement learning in simple words.\n\nAssistant:",
    "Human: Give three tips for effective teamwork.\n\nAssistant:",
    "Human: How can a student manage time better during exams?\n\nAssistant:",
    "Human: What are the benefits of exercise for students?\n\nAssistant:",
    "Human: Describe artificial intelligence in one short paragraph.\n\nAssistant:"
]

print("Number of evaluation prompts:", len(test_prompts))

Number of evaluation prompts: 5


In [24]:
def generate_response(model, prompt, tokenizer, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [27]:
def get_reward_score(text, reward_tokenizer, reward_model, max_length=256):
    inputs = reward_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        logits = reward_model(**inputs).logits

        # If model returns shape [1, 1]
        if logits.shape[-1] == 1:
            score = logits.squeeze().item()
        else:
            # If model returns shape [1, 2] or more, use first value
            score = logits[0][0].item()

    return score

In [28]:
results = []

for prompt in test_prompts:
    baseline_output = generate_response(baseline_model, prompt, tokenizer)
    dpo_output = generate_response(dpo_model, prompt, tokenizer)

    baseline_score = get_reward_score(baseline_output, reward_tokenizer, reward_model)
    dpo_score = get_reward_score(dpo_output, reward_tokenizer, reward_model)

    winner = "DPO" if dpo_score > baseline_score else "Baseline"

    results.append({
        "prompt": prompt,
        "baseline_output": baseline_output,
        "dpo_output": dpo_output,
        "baseline_reward_score": baseline_score,
        "dpo_reward_score": dpo_score,
        "winner": winner
    })

print("Evaluation completed.")

Evaluation completed.


In [29]:
results_df = pd.DataFrame(results)
results_df[["prompt", "baseline_reward_score", "dpo_reward_score", "winner"]]

,prompt,baseline_reward_score,dpo_reward_score,winner
0,Human: Explain reinforcement learning in simpl...,-0.020589,-0.021587,Baseline
1,Human: Give three tips for effective teamwork....,0.008051,-0.013288,Baseline
2,Human: How can a student manage time better du...,0.008554,-0.038127,Baseline
3,Human: What are the benefits of exercise for s...,0.034747,0.018628,Baseline
4,Human: Describe artificial intelligence in one...,0.006215,0.070491,DPO


In [30]:
print(results_df[["prompt", "baseline_reward_score", "dpo_reward_score", "winner"]].to_string(index=False))

                                                                       prompt  baseline_reward_score  dpo_reward_score   winner
         Human: Explain reinforcement learning in simple words.\n\nAssistant:              -0.020589         -0.021587 Baseline
                 Human: Give three tips for effective teamwork.\n\nAssistant:               0.008051         -0.013288 Baseline
      Human: How can a student manage time better during exams?\n\nAssistant:               0.008554         -0.038127 Baseline
         Human: What are the benefits of exercise for students?\n\nAssistant:               0.034747          0.018628 Baseline
Human: Describe artificial intelligence in one short paragraph.\n\nAssistant:               0.006215          0.070491      DPO


In [31]:
dpo_wins = (results_df["winner"] == "DPO").sum()
baseline_wins = (results_df["winner"] == "Baseline").sum()

print("DPO wins:", dpo_wins)
print("Baseline wins:", baseline_wins)

if dpo_wins > baseline_wins:
    print("Overall, the DPO aligned model performed better according to the reward model.")
elif baseline_wins > dpo_wins:
    print("Overall, the baseline model performed better according to the reward model.")
else:
    print("Both models performed equally on this evaluation set.")

DPO wins: 1
Baseline wins: 4
Overall, the baseline model performed better according to the reward model.


In [32]:
for i, row in results_df.iterrows():
    print("\n" + "=" * 100)
    print(f"Example {i+1}")
    print("=" * 100)
    print("\nPROMPT:\n")
    print(row["prompt"])
    print("\nBASELINE OUTPUT:\n")
    print(row["baseline_output"])
    print("\nBASELINE REWARD SCORE:", row["baseline_reward_score"])
    print("\nDPO OUTPUT:\n")
    print(row["dpo_output"])
    print("\nDPO REWARD SCORE:", row["dpo_reward_score"])
    print("\nWINNER:", row["winner"])


Example 1

PROMPT:

Human: Explain reinforcement learning in simple words.

Assistant:

BASELINE OUTPUT:

Human: Explain reinforcement learning in simple words.

Assistant: See examples of reinforcement learning in the following table:
To learn from a simple example, imagine a person who says, "If I can remember to take a few steps to get a few steps, I can learn to write a single sentence."
If someone says, "What are you doing now?" then the person who says, "I know you need to write a single sentence."
You

BASELINE REWARD SCORE: -0.02058943174779415

DPO OUTPUT:

Human: Explain reinforcement learning in simple words.

Assistant: Explain reinforcement learning in simple words.
Assistant: Explain reinforcement learning in simple words.
Assistant: Explain reinforcement learning in simple words.
Assistant: Explain reinforcement learning in simple words.
Assistant: Explain reinforcement learning in simple words.
Assistant: Explain reinforcement learning in simple words.
Assistant: Expla

In [34]:
results_df.to_csv("evaluation_results.csv", index=False)
print("Results saved to evaluation_results.csv")

Results saved to evaluation_results.csv
